# 面试题：人类审批怎样绑定具体动作？

可复述答案：审批票据绑定动作摘要、资源、金额、租户、发起人、策略版本、过期与一次性 nonce。执行端重新计算摘要，检查审批人权限、票据未使用及当前业务前置条件。只存一个 approved 布尔值无法阻止金额被改后重放，也无法审计人究竟批准了什么。

## 真实案例

采购 Agent 请求经理审批设备订单。六个事件包含同金额、金额变更、过期、nonce 重放、审批人无权限和拒绝。

## 基线

基线只要请求带 approved=True 就提交订单。

## 结果解读

手写 verifier 输出摘要、过期、审批角色与 nonce 使用状态。

## 失败案例

审批后把金额从 800 改为 1200，旧票据必须失效。

In [1]:
events = [{'id':'H1','order':'PO1','amount':800,'approved':True,'ticket_amount':800,'role':'manager','expired':False,'nonce':'n1'}, {'id':'H2','order':'PO2','amount':1200,'approved':True,'ticket_amount':800,'role':'manager','expired':False,'nonce':'n2'}, {'id':'H3','order':'PO3','amount':500,'approved':True,'ticket_amount':500,'role':'manager','expired':True,'nonce':'n3'}, {'id':'H4','order':'PO1','amount':800,'approved':True,'ticket_amount':800,'role':'manager','expired':False,'nonce':'n1'}, {'id':'H5','order':'PO4','amount':300,'approved':True,'ticket_amount':300,'role':'employee','expired':False,'nonce':'n5'}, {'id':'H6','order':'PO5','amount':200,'approved':False,'ticket_amount':200,'role':'manager','expired':False,'nonce':'n6'}]  # 构造六条审批与执行事件，覆盖篡改、过期、重放和角色错误。
print('审批事件:', events)  # 输出执行端收到的动作和审批票据字段。
print('教学说明：所有 nonce 与订单号均为离线脱敏值，真实系统需签名或服务端票据存储。')  # 说明示例数据边界。

审批事件: [{'id': 'H1', 'order': 'PO1', 'amount': 800, 'approved': True, 'ticket_amount': 800, 'role': 'manager', 'expired': False, 'nonce': 'n1'}, {'id': 'H2', 'order': 'PO2', 'amount': 1200, 'approved': True, 'ticket_amount': 800, 'role': 'manager', 'expired': False, 'nonce': 'n2'}, {'id': 'H3', 'order': 'PO3', 'amount': 500, 'approved': True, 'ticket_amount': 500, 'role': 'manager', 'expired': True, 'nonce': 'n3'}, {'id': 'H4', 'order': 'PO1', 'amount': 800, 'approved': True, 'ticket_amount': 800, 'role': 'manager', 'expired': False, 'nonce': 'n1'}, {'id': 'H5', 'order': 'PO4', 'amount': 300, 'approved': True, 'ticket_amount': 300, 'role': 'employee', 'expired': False, 'nonce': 'n5'}, {'id': 'H6', 'order': 'PO5', 'amount': 200, 'approved': False, 'ticket_amount': 200, 'role': 'manager', 'expired': False, 'nonce': 'n6'}]
教学说明：所有 nonce 与订单号均为离线脱敏值，真实系统需签名或服务端票据存储。


In [2]:
baseline = [(row['id'], '提交' if row['approved'] else '拒绝') for row in events]  # 构造只信任布尔审批字段的危险基线。
print('布尔审批基线:', baseline)  # 输出基线会错误放过金额变更和重放。
print('基线风险：它完全没有绑定订单、金额、角色、有效期与一次性语义。')  # 点出审批票据必须携带的约束。

布尔审批基线: [('H1', '提交'), ('H2', '提交'), ('H3', '提交'), ('H4', '提交'), ('H5', '提交'), ('H6', '拒绝')]
基线风险：它完全没有绑定订单、金额、角色、有效期与一次性语义。


In [3]:
used_nonces = set()  # 初始化已消费审批票据 nonce 的服务端账本。
def verify_approval(row):  # 定义执行前的审批票据验证器。
    if not row['approved']:  # 检查审批是否明确同意。
        return 'rejected:not_approved'  # 处理明确拒绝。
    if row['role'] != 'manager':  # 检查签发人是否具备审批权限。
        return 'rejected:role'  # 拒绝普通员工伪造的同意字段。
    if row['expired']:  # 检查票据是否仍在有效期内。
        return 'rejected:expired'  # 阻止旧决策在条件变化后重放。
    if row['amount'] != row['ticket_amount']:  # 重新比对动作金额与被批准金额。
        return 'rejected:digest_mismatch'  # 阻止审批后篡改动作摘要。
    if row['nonce'] in used_nonces:  # 检查票据是否已被一次性消费。
        return 'rejected:replay'  # 阻止相同审批重复提交。
    used_nonces.add(row['nonce'])  # 原子标记当前票据已消费。
    return 'approved_for_execute'  # 返回可进入后续领域校验的结论。

In [4]:
results = [(row['id'], verify_approval(row)) for row in events]  # 按到达顺序验证六张审批票据。
print('id | 审批验证结论')  # 输出票据门禁结果表标题。
for item in results:  # 遍历每个请求的明确通过或拒绝原因。
    print(item[0], item[1])  # 输出面向审计的结构化结果。
print('实际可执行数:', sum(status == 'approved_for_execute' for _, status in results))  # 汇总真正绑定到动作的有效审批数。

id | 审批验证结论
H1 approved_for_execute
H2 rejected:digest_mismatch
H3 rejected:expired
H4 rejected:replay
H5 rejected:role
H6 rejected:not_approved
实际可执行数: 1


In [5]:
wrong = dict(baseline)['H2']  # 读取金额被篡改时布尔基线的错误结论。
fixed = dict(results)['H2']  # 读取摘要绑定验证后的正确拒绝。
print('失败案例 H2：布尔基线=', wrong, '，票据验证=', fixed)  # 展示审批不绑定动作会产生的越权风险。
print('生产差距：还需身份提供方、票据签名、策略版本、撤销、最小作用域、审批 UI diff 与完整审计。')  # 说明真实审批系统的控制面。

失败案例 H2：布尔基线= 提交 ，票据验证= rejected:digest_mismatch
生产差距：还需身份提供方、票据签名、策略版本、撤销、最小作用域、审批 UI diff 与完整审计。


In [6]:
assert dict(results)['H1'] == 'approved_for_execute'  # 验证完全匹配且未使用的经理票据可通过。
assert dict(results)['H2'] == 'rejected:digest_mismatch'  # 验证金额变化会使旧审批失效。
assert dict(results)['H4'] == 'rejected:replay'  # 验证 nonce 重放不能重复执行。